# 07 - Benchmark Analysis & Automated Statistical Reporting

This notebook provides a complete visual and statistical evaluation of the **LLaMEA Champion** against classical baseline optimizers (**CMA-ES**, **DE**, **PSO**).

## 🎯 Conceptual Workflow & Research Questions
For every target BBOB problem across evaluated dimensions and noise levels, we address two fundamental scientific questions:

```
                         4 Optimization Algorithms
                     (LLaMEA Champion, CMA-ES, DE, PSO)
                                     │
                                     ▼
                       What is the research question?
                                     │
                 ┌───────────────────┴───────────────────┐
                 ▼                                       ▼
    [Question 1: Overall Difference]        [Question 2: Pairwise Superiority]
   "Are any algorithms different?"         "Is LLaMEA better than baseline X?"
                 │                                       │
                 ▼                                       ▼
       Kruskal-Wallis H Test                 Mann-Whitney U + Vargha-Delaney A12
```

### 📊 The Statistical Inference Chain
- **Sample Statistic**: Optimization runs are stochastic random variables. Rather than comparing simple means (which assume normality), we analyze the full distribution of terminal residuals (final minimum error $\min y$, where **LOWER is BETTER**).
- **Sampling Distribution**: Rank-based non-parametric tests pool observations, rank them from best to worst, and assess whether rank distributions deviate from random chance under the Null Hypothesis ($H_0$).
- **p-value**: Probability of obtaining empirical rank sums as extreme as observed, assuming no difference between algorithms.
- **Effect Size ($\hat{A}_{12}$)**: Vargha-Delaney metric quantifying the probability that LLaMEA achieves a lower residual than a baseline ($A_{12} < 0.5$ indicates LLaMEA superiority).

---
### 📁 Automated Artifact Outputs in `results/`
Running this notebook automatically populates the consolidated `results/` directory:
1. **High-Res Figures**: `results/figures/{dim}D/std_{noise_std}/f{p_id}.png` (Dual-panel: Convergence trajectory + ECDF).
2. **Tabular CSV Reports**: `results/reports/statistical_summary.csv` & `results/reports/kruskal_wallis_summary.csv`.
3. **Publication Markdown Report**: `results/reports/statistical_summary.md` (Rich formatted tables, function classes, and badges).

## 1. Setup, Environment & Configuration

### 🎯 What this cell does:
Imports necessary scientific and plotting libraries, configures root directory paths (`PROJECT_ROOT`, `IOH_LOGS_DIR`, `RESULTS_DIR`), and defines benchmark target metadata (problems, dimensions, noise levels, and visualization color schemes).

In [6]:
import io
import os
import re
import sys
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display

# ── Consolidated Path Configuration ───────────────────────
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
IOH_LOGS_DIR = PROJECT_ROOT / 'data' / 'ioh_logs'
RESULTS_DIR  = PROJECT_ROOT / 'results'
FIGURES_DIR  = RESULTS_DIR / 'figures'
REPORTS_DIR  = RESULTS_DIR / 'reports'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05  # Statistical significance threshold

# BBOB Metadata Mapping for High-Readability Reporting
BBOB_NAMES = {
    1:  'Sphere (f1)',
    8:  'Rosenbrock (f8)',
    11: 'Discus (f11)',
    15: 'Rastrigin (f15)',
    21: 'Gallagher 101 Peaks (f21)'
}

BBOB_CLASSES = {
    1:  'Separable',
    8:  'Low Conditioning',
    11: 'High Conditioning',
    15: 'Multi-Modal (Global Struct)',
    21: 'Multi-Modal (Weak Struct)'
}

ALGO_COLORS = {
    'LLaMEA Champion': '#d62728',  # Vibrant Crimson
    'LLaMEA':          '#d62728',
    'CMA-ES':          '#1f77b4',  # Royal Blue
    'DE':              '#2ca02c',  # Emerald Green
    'PSO':             '#ff7f0e'   # Amber Orange
}

def hex_to_rgba(hex_color: str, alpha: float = 0.15) -> str:
    """Convert hex color string to rgba CSS string."""
    hex_color = hex_color.lstrip('#')
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f'rgba({r}, {g}, {b}, {alpha})'

print(f'IOH Logs Source:   {IOH_LOGS_DIR}')
print(f'Consolidated Out:  {RESULTS_DIR}')
print(f'  ├── Figures:     {FIGURES_DIR}')
print(f'  └── Reports:     {REPORTS_DIR}')


IOH Logs Source:   /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/ioh_logs
Consolidated Out:  /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results
  ├── Figures:     /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures
  └── Reports:     /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/reports
Target Problems:   f[1, 8, 11, 15, 21]
Dimensions:        [2, 3]
Noise STDs:        [0.0, 0.05, 0.1]


## 2. Robust IOH Data Ingestion Helpers

### 🎯 What this cell does:
Defines parsing functions to load IOHanalyzer `.dat` trajectory files from disk:
- `parse_ioh_dat_file()`: Splits multi-run `.dat` files into separate run DataFrames containing `evaluations` and `raw_y`.
- `load_problem_benchmark_data()`: Scans algorithm directories for a specific problem, standardizing algorithm display names.
- `extract_terminal_residuals()`: Extracts the lowest objective error (terminal residual $\min y$) achieved across each independent run.

In [7]:
def parse_ioh_dat_file(dat_path: Path) -> list[pd.DataFrame]:
    """Parse an IOH .dat file into a list of run DataFrames containing evaluations and raw_y."""
    if not dat_path.exists() or dat_path.stat().st_size == 0:
        return []
        
    text = dat_path.read_text(encoding='utf-8', errors='ignore').strip()
    if not text:
        return []
        
    runs = []
    blocks = [b.strip() for b in text.split('evaluations') if b.strip()]
    
    for block in blocks:
        lines = [l for l in block.splitlines() if l.strip() and not l.strip().startswith('#') and not l.strip().startswith('raw_y')]
        if not lines:
            continue
        csv_data = 'evaluations raw_y' + chr(10) + chr(10).join(lines)
        try:
            df = pd.read_csv(io.StringIO(csv_data), sep=r'\s+')
            df['evaluations'] = pd.to_numeric(df['evaluations'], errors='coerce')
            df['raw_y'] = pd.to_numeric(df['raw_y'], errors='coerce')
            df = df.dropna().reset_index(drop=True)
            if len(df) > 0:
                runs.append(df)
        except Exception:
            pass
            
    return runs


def load_problem_benchmark_data(problem_dir: Path) -> dict[str, list[pd.DataFrame]]:
    """Load all algorithm trajectories from a problem directory, standardized by display name."""
    data = {}
    if not problem_dir.exists():
        return data
        
    for algo_dir in sorted(problem_dir.iterdir()):
        if not algo_dir.is_dir():
            continue
            
        base_name = re.sub(r'-\d+$', '', algo_dir.name)
        clean_name = base_name.split('_std')[0]
        
        # Ignore test dummies or non-target temporary logs
        if any(dk in clean_name for dk in ["dummy-llm", "failing-llm", "local-model"]):
            continue
            
        if 'llamea_champion' in clean_name or 'champion' in clean_name:
            display_name = 'LLaMEA Champion'
        elif 'llamea' in clean_name or 'qwen' in clean_name or 'llama' in clean_name or 'claude' in clean_name or 'gpt' in clean_name:
            clean_model = clean_name.replace("llamea_", "").split("_exp")[0].removesuffix(".gguf").removesuffix(".bin").replace("-instruct", "").replace("_q4_k_m", "")
            display_name = f'LLaMEA ({clean_model})'
        else:
            display_name = clean_name.upper()
            
        runs = []
        for dat_file in algo_dir.rglob('*.dat'):
            runs.extend(parse_ioh_dat_file(dat_file))
                
        if runs:
            if display_name in data:
                data[display_name].extend(runs)
            else:
                data[display_name] = runs
                
    return data


def extract_terminal_residuals(algo_runs: dict[str, list[pd.DataFrame]]) -> dict[str, list[float]]:
    """Extract terminal residuals (minimum raw_y achieved per run)."""
    residuals = {}
    for algo, runs in algo_runs.items():
        residuals[algo] = [float(df['raw_y'].min()) for df in runs if not df.empty]
    return residuals

print('Data loading and extraction functions ready.')

Data loading and extraction functions ready.


## 3. Data Loading & Coverage Verification

### 🎯 What this cell does:
Iterates through the target dimensions and noise levels in `data/ioh_logs/` and loads all available benchmark runs into an in-memory dictionary `all_benchmark_data`. Displays a quick summary of available runs per problem and algorithm.

In [8]:
all_benchmark_data = {}
summary_records = []

# Dynamically discover all evaluated (dim, noise_std, problem_id) from ioh_logs
for dim_dir in sorted(IOH_LOGS_DIR.glob('*D')):
    if not dim_dir.is_dir():
        continue
    dim_str = dim_dir.name.replace('D', '')
    try:
        dim = int(dim_str)
    except ValueError:
        continue
        
    for std_dir in sorted(dim_dir.glob('std_*')):
        if not std_dir.is_dir():
            continue
        try:
            noise_std = float(std_dir.name.replace('std_', ''))
        except ValueError:
            continue
            
        for prob_dir in sorted(std_dir.glob('f*')):
            if not prob_dir.is_dir():
                continue
            try:
                p_id = int(prob_dir.name.replace('f', ''))
            except ValueError:
                continue
                
            algo_runs = load_problem_benchmark_data(prob_dir)
            if algo_runs:
                all_benchmark_data[(dim, noise_std, p_id)] = algo_runs
                for algo, runs in algo_runs.items():
                    summary_records.append({
                        'Dim': f"{dim}D",
                        'Noise Std': noise_std,
                        'Problem': BBOB_NAMES.get(p_id, f'f{p_id}'),
                        'Algorithm': algo,
                        'Runs': len(runs)
                    })

df_coverage = pd.DataFrame(summary_records)
print(f"🎯 Dynamically loaded benchmark data for {len(all_benchmark_data)} unique problem configurations from disk.")
if not df_coverage.empty:
    display(df_coverage.groupby(['Dim', 'Noise Std', 'Problem', 'Algorithm'])['Runs'].sum().unstack(fill_value=0))


Loaded benchmark data for 30 unique problem configurations.


Algorithm                                CMAES  DE  LLaMEA Champion  PSO
Dim Noise Std Problem                                                   
2D  0.00      Discus (f11)                  10  10               10   10
              Gallagher 101 Peaks (f21)     10  10               10   10
              Rastrigin (f15)               10  10               10   10
              Rosenbrock (f8)               10  10               10   10
              Sphere (f1)                   10  10               10   10
    0.05      Discus (f11)                  10  10               10   10
              Gallagher 101 Peaks (f21)     10  10               10   10
              Rastrigin (f15)               10  10               10   10
              Rosenbrock (f8)               10  10               10   10
              Sphere (f1)                   10  10               10   10
    0.10      Discus (f11)                  10  10                0   10
              Gallagher 101 Peaks (f21)     10  10                0   10
              Rastrigin (f15)               10  10                0   10
              Rosenbrock (f8)               10  10                0   10
              Sphere (f1)                   10  10                0   10
3D  0.00      Discus (f11)                  10  10               10   10
              Gallagher 101 Peaks (f21)     10  10               10   10
              Rastrigin (f15)               10  10               10   10
              Rosenbrock (f8)               10  10               10   10
              Sphere (f1)                   10  10               10   10
    0.05      Discus (f11)                  10  10               10   10
              Gallagher 101 Peaks (f21)     10  10               10   10
              Rastrigin (f15)               10  10               10   10
              Rosenbrock (f8)               10  10               10   10
              Sphere (f1)                   10  10               10   10
    0.10      Discus (f11)                  10  10                0   10
              Gallagher 101 Peaks (f21)     10  10                0   10
              Rastrigin (f15)               10  10                0   10
              Rosenbrock (f8)               10  10                0   10
              Sphere (f1)                   10  10                0   10

## 4. Dual-Panel Benchmark Figures Generation (Convergence & ECDF)

### 🎯 What this cell does:
For every loaded problem setting, generates a dual-panel Plotly figure and exports a high-resolution PNG to `results/figures/{dim}D/std_{noise_std}/f{p_id}.png`:
- **Left Panel (Convergence Curves)**: Interpolates the best-so-far clean error across log-spaced evaluation steps ($10^0$ to $10^5$) and renders mean trajectories with shaded $\pm 1\sigma$ error bands.
- **Right Panel (ECDF Curves)**: Computes the Empirical Cumulative Distribution Function over target precision thresholds $\tau \in [10^{-8}, 10^2]$, visualizing the fraction of trials reaching each target precision.

In [9]:
for (dim, noise_std, p_id), algo_runs in all_benchmark_data.items():
    p_name = BBOB_NAMES.get(p_id, f'f{p_id}')
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'{p_name} ({dim}D, Noise std={noise_std}) - Convergence',
            f'{p_name} ({dim}D, Noise std={noise_std}) - ECDF'
        ),
        horizontal_spacing=0.10
    )
    
    # --- Subplot 1: Fixed-Budget Convergence ---
    eval_grid = np.logspace(0, 5, 200)
    for algo_name, runs in algo_runs.items():
        color = ALGO_COLORS.get(algo_name, '#7f7f7f')
        rgba_color = hex_to_rgba(color, 0.15)
        
        interp_y = []
        for df in runs:
            evals = df['evaluations'].values
            errors = df['raw_y'].values
            cum_min_errors = np.minimum.accumulate(errors)
            y_interp = np.interp(eval_grid, evals, cum_min_errors, left=cum_min_errors[0], right=cum_min_errors[-1])
            interp_y.append(y_interp)
            
        mean_y = np.mean(interp_y, axis=0)
        std_y = np.std(interp_y, axis=0)
        upper_y = mean_y + std_y
        lower_y = np.maximum(mean_y - std_y, 1e-12)
        
        # Upper confidence band
        fig.add_trace(
            go.Scatter(x=eval_grid, y=upper_y, mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'),
            row=1, col=1
        )
        # Lower confidence band with fill
        fig.add_trace(
            go.Scatter(x=eval_grid, y=lower_y, mode='lines', line=dict(width=0), fill='tonexty', fillcolor=rgba_color, showlegend=False, hoverinfo='skip'),
            row=1, col=1
        )
        # Mean trajectory
        fig.add_trace(
            go.Scatter(x=eval_grid, y=mean_y, mode='lines', name=algo_name, line=dict(color=color, width=2.5), hovertemplate='Evals: %{x:.0f}<br>Error: %{y:.3e}'),
            row=1, col=1
        )
        
    # --- Subplot 2: Empirical Cumulative Distribution Function (ECDF) ---
    targets = np.logspace(-8, 2, 100)
    for algo_name, runs in algo_runs.items():
        color = ALGO_COLORS.get(algo_name, '#7f7f7f')
        hit_rates = []
        for t in targets:
            hits = sum(1 for df in runs if np.min(df['raw_y'].values) <= t)
            hit_rates.append(hits / len(runs))
            
        fig.add_trace(
            go.Scatter(x=targets, y=hit_rates, mode='lines', name=algo_name, line=dict(color=color, width=2.5), showlegend=False, hovertemplate='Target τ: %{x:.2e}<br>Fraction: %{y:.2f}'),
            row=1, col=2
        )
        
    fig.update_xaxes(type='log', title_text='Function Evaluations', row=1, col=1, gridcolor='#EBEBEB')
    fig.update_yaxes(type='log', title_text='Clean Distance to Optimum Δf(x)', row=1, col=1, gridcolor='#EBEBEB')
    fig.update_xaxes(type='log', title_text='Target Precision τ', row=1, col=2, gridcolor='#EBEBEB')
    fig.update_yaxes(title_text='ECDF (Success Rate)', range=[0, 1.05], row=1, col=2, gridcolor='#EBEBEB')
    
    fig.update_layout(
        template='plotly_white',
        font=dict(family='sans-serif', size=12),
        legend=dict(orientation='h', yanchor='bottom', y=1.12, xanchor='center', x=0.5, bgcolor='rgba(255,255,255,0.8)', bordercolor='#E0E0E0', borderwidth=1),
        margin=dict(t=90, b=50, l=60, r=40),
        height=450,
        width=980
    )
    
    # Save figure to results/figures/{dim}D/std_{noise_std}/f{p_id}.png
    out_fig_dir = FIGURES_DIR / f"{dim}D" / f"std_{noise_std}"
    out_fig_dir.mkdir(parents=True, exist_ok=True)
    png_path = out_fig_dir / f"f{p_id}.png"
    fig.write_image(str(png_path), scale=2)
    print(f"  ✅ Figure saved: {png_path.relative_to(PROJECT_ROOT)}")

print('\n✨ All benchmark convergence and ECDF figures successfully exported to results/figures/.')

  ✅ Figure saved: results/figures/2D/std_0.0/f1.png
  ✅ Figure saved: results/figures/2D/std_0.0/f8.png
  ✅ Figure saved: results/figures/2D/std_0.0/f11.png
  ✅ Figure saved: results/figures/2D/std_0.0/f15.png
  ✅ Figure saved: results/figures/2D/std_0.0/f21.png
  ✅ Figure saved: results/figures/2D/std_0.05/f1.png
  ✅ Figure saved: results/figures/2D/std_0.05/f8.png
  ✅ Figure saved: results/figures/2D/std_0.05/f11.png
  ✅ Figure saved: results/figures/2D/std_0.05/f15.png
  ✅ Figure saved: results/figures/2D/std_0.05/f21.png
  ✅ Figure saved: results/figures/2D/std_0.1/f1.png
  ✅ Figure saved: results/figures/2D/std_0.1/f8.png
  ✅ Figure saved: results/figures/2D/std_0.1/f11.png
  ✅ Figure saved: results/figures/2D/std_0.1/f15.png
  ✅ Figure saved: results/figures/2D/std_0.1/f21.png
  ✅ Figure saved: results/figures/3D/std_0.0/f1.png
  ✅ Figure saved: results/figures/3D/std_0.0/f8.png
  ✅ Figure saved: results/figures/3D/std_0.0/f11.png
  ✅ Figure saved: results/figures/3D/std_0.0/f15.

## 5. Statistical Testing Core (Non-Parametric & Effect Size)

### 🎯 What this cell does:
Defines the statistical hypothesis testing functions:
- `vargha_delaney_A12()`: Calculates the Vargha-Delaney non-parametric effect size. In minimization (lower is better), $\hat{A}_{12} < 0.5$ indicates LLaMEA superiority.
- `classify_effect_magnitude()`: Maps $\hat{A}_{12}$ to standard effect size categories (*Negligible*, *Small*, *Medium*, *Large*).
- `run_omnibus_kruskal_wallis()`: Performs the Kruskal-Wallis H-Test across all algorithms to test for significant global differences.
- `run_pairwise_tests()`: Executes two-sided Mann-Whitney U tests paired with $\hat{A}_{12}$ effect sizes for LLaMEA Champion vs. each baseline.

In [10]:
def vargha_delaney_A12(m: list[float] | np.ndarray, n: list[float] | np.ndarray) -> float:
    """
    Compute Vargha-Delaney A12 effect size statistic.
    A12 measures the probability that a random run from group m obtains a higher residual
    than a random run from group n (with ties broken evenly).
    
    In minimization (LOWER is BETTER):
      - A12 < 0.5  -> Algorithm 1 (LLaMEA) is superior (yields lower residuals)
      - A12 = 0.5  -> Algorithms are equivalent (identical rank distribution)
      - A12 > 0.5  -> Algorithm 2 (Baseline) is superior
    """
    m = np.asarray(m, dtype=float)
    n = np.asarray(n, dtype=float)
    m_len, n_len = len(m), len(n)
    
    if m_len == 0 or n_len == 0:
        return 0.5
        
    # Check for identical values across both groups
    if np.all(m == m[0]) and np.all(n == n[0]) and m[0] == n[0]:
        return 0.5
        
    ranks = stats.rankdata(np.concatenate([m, n]))
    r1 = np.sum(ranks[:m_len])
    a12 = (r1 / m_len - (m_len + 1) / 2.0) / n_len
    return float(a12)


def classify_effect_magnitude(a12: float) -> str:
    """Classify Vargha-Delaney A12 magnitude based on standard thresholds."""
    d = abs(a12 - 0.5)
    if d < 0.06:
        return 'Negligible'
    elif d < 0.14:
        return 'Small'
    elif d < 0.21:
        return 'Medium'
    else:
        return 'Large'


def run_omnibus_kruskal_wallis(residuals_dict: dict[str, list[float]], alpha: float = ALPHA):
    """Run Kruskal-Wallis H Test to detect overall differences across all algorithms."""
    valid_samples = [s for s in residuals_dict.values() if len(s) > 0]
    if len(valid_samples) < 2:
        return None, None, False
        
    # If all samples across all algorithms are completely identical, H=0 and p=1.0
    all_values = np.concatenate([np.asarray(s) for s in valid_samples])
    if np.all(all_values == all_values[0]):
        return 0.0, 1.0, False
        
    try:
        h_stat, p_val = stats.kruskal(*valid_samples)
        return float(h_stat), float(p_val), bool(p_val < alpha)
    except Exception:
        return 0.0, 1.0, False


def run_pairwise_tests(
    residuals_dict: dict[str, list[float]], 
    target_algo: str = 'LLaMEA Champion',
    alpha: float = ALPHA
) -> pd.DataFrame:
    """Run pairwise Mann-Whitney U tests and compute Vargha-Delaney A12 effect sizes."""
    if target_algo not in residuals_dict or len(residuals_dict[target_algo]) == 0:
        return pd.DataFrame()
        
    target_samples = np.asarray(residuals_dict[target_algo], dtype=float)
    records = []
    
    for algo, baseline_samples in residuals_dict.items():
        if algo == target_algo or len(baseline_samples) == 0:
            continue
            
        baseline_samples = np.asarray(baseline_samples, dtype=float)
        
        # Check if both groups are completely identical
        if np.all(target_samples == target_samples[0]) and np.all(baseline_samples == baseline_samples[0]) and target_samples[0] == baseline_samples[0]:
            p_val = 1.0
            is_sig = False
            a12 = 0.5
            magnitude = 'Negligible'
            outcome = 'Exact Tie'
        else:
            try:
                stat, p_val = stats.mannwhitneyu(target_samples, baseline_samples, alternative='two-sided')
            except Exception:
                p_val = 1.0
            a12 = vargha_delaney_A12(target_samples, baseline_samples)
            magnitude = classify_effect_magnitude(a12)
            is_sig = bool(p_val < alpha)
            
            # Lower residual is better
            if is_sig and a12 < 0.5:
                outcome = 'LLaMEA Wins (Sig)'
            elif is_sig and a12 > 0.5:
                outcome = f'{algo} Wins (Sig)'
            elif a12 < 0.5:
                outcome = 'LLaMEA Ahead (Non-Sig)'
            elif a12 > 0.5:
                outcome = f'{algo} Ahead (Non-Sig)'
            else:
                outcome = 'Exact Tie'
            
        records.append({
            'Baseline': algo,
            'LLaMEA Mean Res': float(np.mean(target_samples)),
            'LLaMEA Med Res': float(np.median(target_samples)),
            'Baseline Mean Res': float(np.mean(baseline_samples)),
            'Baseline Med Res': float(np.median(baseline_samples)),
            'Mann-Whitney p-val': float(p_val),
            'Significant (α=0.05)': 'Yes' if is_sig else 'No',
            'A12 Effect Size': float(a12),
            'Effect Magnitude': magnitude,
            'Outcome': outcome
        })
        
    return pd.DataFrame(records)

print('Statistical testing functions compiled.')

Statistical testing functions compiled.


## 6. Execute Hypothesis Testing & Effect Size Battery

### 🎯 What this cell does:
Runs the statistical tests on the terminal residuals across all loaded problems:
1. Computes the Kruskal-Wallis $H$-statistic and $p$-value for global differences.
2. Computes pairwise Mann-Whitney $U$ $p$-values and Vargha-Delaney $\hat{A}_{12}$ effect sizes comparing LLaMEA Champion against each baseline.

In [11]:
master_pairwise_results = []
master_omnibus_results = []

for (dim, noise_std, p_id), algo_runs in all_benchmark_data.items():
    p_name = BBOB_NAMES.get(p_id, f'f{p_id}')
    p_class = BBOB_CLASSES.get(p_id, 'Benchmark')
    
    residuals = extract_terminal_residuals(algo_runs)
    
    # 1. Omnibus Kruskal-Wallis
    h_stat, kw_pval, is_sig = run_omnibus_kruskal_wallis(residuals)
    if h_stat is not None:
        master_omnibus_results.append({
            'Dim': dim,
            'Noise Std': noise_std,
            'Problem ID': f'f{p_id}',
            'Problem Name': p_name,
            'Function Class': p_class,
            'H-Statistic': h_stat,
            'p-value': kw_pval,
            'Significant Difference': 'Yes' if is_sig else 'No'
        })
    
    # 2. Pairwise LLaMEA Champion vs Baselines
    df_pair = run_pairwise_tests(residuals, target_algo='LLaMEA Champion')
    if not df_pair.empty:
        df_pair.insert(0, 'Function Class', p_class)
        df_pair.insert(0, 'Problem Name', p_name)
        df_pair.insert(0, 'Problem ID', f'f{p_id}')
        df_pair.insert(0, 'Noise Std', noise_std)
        df_pair.insert(0, 'Dim', dim)
        master_pairwise_results.append(df_pair)

print(f'Completed statistical battery across {len(master_omnibus_results)} problem settings.')

Completed statistical battery across 30 problem settings.


## 7. Interactive Results Inspection

### 🎯 What this cell does:
Combines all pairwise results into master DataFrames and displays the interactive summary tables directly in the notebook for immediate inspection.

In [12]:
df_all_pairwise = pd.concat(master_pairwise_results, ignore_index=True) if master_pairwise_results else pd.DataFrame()
df_all_omnibus  = pd.DataFrame(master_omnibus_results)

print('=== 1. Omnibus Kruskal-Wallis Group Differences ===')
display(df_all_omnibus)

print('\n=== 2. Pairwise LLaMEA Champion vs Baselines ===')
if not df_all_pairwise.empty:
    display(df_all_pairwise[['Dim', 'Noise Std', 'Problem Name', 'Function Class', 'Baseline', 'LLaMEA Med Res', 'Baseline Med Res', 'Mann-Whitney p-val', 'A12 Effect Size', 'Effect Magnitude', 'Outcome']])

=== 1. Omnibus Kruskal-Wallis Group Differences ===


,Dim,Noise Std,Problem ID,Problem Name,Function Class,H-Statistic,p-value,Significant Difference
0,2,0.00,f1,Sphere (f1),Separable,0.000000,1.000000e+00,No
1,2,0.00,f8,Rosenbrock (f8),Low Conditioning,34.866238,1.300154e-07,Yes
2,2,0.00,f11,Discus (f11),High Conditioning,26.444654,7.696691e-06,Yes
3,2,0.00,f15,Rastrigin (f15),Multi-Modal (Global Struct),16.488982,9.000719e-04,Yes
4,2,0.00,f21,Gallagher 101 Peaks (f21),Multi-Modal (Weak Struct),9.291578,2.565519e-02,Yes
5,2,0.05,f1,Sphere (f1),Separable,37.956204,2.887467e-08,Yes
6,2,0.05,f8,Rosenbrock (f8),Low Conditioning,35.854920,8.036368e-08,Yes
7,2,0.05,f11,Discus (f11),High Conditioning,8.862396,3.117747e-02,Yes
8,2,0.05,f15,Rastrigin (f15),Multi-Modal (Global Struct),17.594407,5.332226e-04,Yes
9,2,0.05,f21,Gallagher 101 Peaks (f21),Multi-Modal (Weak Struct),10.691379,1.351731e-02,Yes



=== 2. Pairwise LLaMEA Champion vs Baselines ===


,Dim,Noise Std,Problem Name,Function Class,Baseline,LLaMEA Med Res,Baseline Med Res,Mann-Whitney p-val,A12 Effect Size,Effect Magnitude,Outcome
0,2,0.00,Sphere (f1),Separable,CMAES,0.000000e+00,0.000000e+00,1.000000,0.50,Negligible,Exact Tie
1,2,0.00,Sphere (f1),Separable,DE,0.000000e+00,0.000000e+00,1.000000,0.50,Negligible,Exact Tie
2,2,0.00,Sphere (f1),Separable,PSO,0.000000e+00,0.000000e+00,1.000000,0.50,Negligible,Exact Tie
3,2,0.00,Rosenbrock (f8),Low Conditioning,CMAES,8.471105e+03,0.000000e+00,0.000055,1.00,Large,CMAES Wins (Sig)
4,2,0.00,Rosenbrock (f8),Low Conditioning,DE,8.471105e+03,3.000000e-10,0.000153,1.00,Large,DE Wins (Sig)
5,2,0.00,Rosenbrock (f8),Low Conditioning,PSO,8.471105e+03,0.000000e+00,0.000055,1.00,Large,PSO Wins (Sig)
6,2,0.00,Discus (f11),High Conditioning,CMAES,1.683900e-06,0.000000e+00,0.000064,1.00,Large,CMAES Wins (Sig)
7,2,0.00,Discus (f11),High Conditioning,DE,1.683900e-06,6.829281e-04,0.002202,0.09,Large,LLaMEA Wins (Sig)
8,2,0.00,Discus (f11),High Conditioning,PSO,1.683900e-06,6.950000e-09,0.088733,0.73,Large,PSO Ahead (Non-Sig)
9,2,0.00,Rastrigin (f15),Multi-Modal (Global Struct),CMAES,1.091657e-01,1.185007e+00,0.025470,0.20,Large,LLaMEA Wins (Sig)


## 8. Automated Multi-Format Report Export (CSV & Markdown)

### 🎯 What this cell does:
Automatically compiles and exports the complete analysis into reproducible report files in `results/reports/`:
1. `results/reports/statistical_summary.csv`: Machine-readable pairwise comparisons.
2. `results/reports/kruskal_wallis_summary.csv`: Machine-readable omnibus tests.
3. `results/reports/statistical_summary.md`: Publication-ready formatted Markdown report with visual outcome badges.

In [13]:
if not df_all_pairwise.empty:
    # ── 1. Export Per-(Dim, Std) Reports Matching ioh_logs Hierarchy ──────────
    grouped = df_all_pairwise.groupby(['Dim', 'Noise Std'])
    
    for (dim, noise_std), df_subset in grouped:
        subset_report_dir = REPORTS_DIR / f"{dim}D" / f"std_{noise_std}"
        subset_report_dir.mkdir(parents=True, exist_ok=True)
        
        # 1a. Pairwise CSV
        pair_csv_path = subset_report_dir / 'statistical_summary.csv'
        df_subset.to_csv(pair_csv_path, index=False)
        
        # 1b. Omnibus CSV
        df_omnibus_subset = df_all_omnibus[(df_all_omnibus['Dim'] == dim) & (df_all_omnibus['Noise Std'] == noise_std)]
        omnibus_csv_path = subset_report_dir / 'kruskal_wallis_summary.csv'
        df_omnibus_subset.to_csv(omnibus_csv_path, index=False)
        
        # 1c. Formatted Markdown Report for this (Dim, Std)
        sub_md_path = subset_report_dir / 'statistical_summary.md'
        noise_title = "Clean Mode (std = 0.0)" if noise_std == 0.0 else f"Noisy Mode (std = {noise_std})"
        
        n_tests = len(df_subset)
        sub_llamea_wins = len(df_subset[df_subset['Outcome'].str.contains('LLaMEA Wins', regex=False)])
        sub_base_wins = len(df_subset[df_subset['Outcome'].str.contains('Wins (Sig)', regex=False) & ~df_subset['Outcome'].str.contains('LLaMEA', regex=False)])
        sub_ties = n_tests - sub_llamea_wins - sub_base_wins
        
        lines = [
            f"# 📊 Benchmark Statistical Summary: {dim}D ({noise_title})",
            "",
            f"> **Target Dimension:** `{dim}D` | **Noise Level:** `{noise_std}`",
            "",
            "## 🏆 Win-Loss Summary",
            f"- **Total Pairwise Tests ($N$):** `{n_tests}`",
            rf"- **🟢 LLaMEA Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} < 0.5$):** **`{sub_llamea_wins}`** ({sub_llamea_wins/n_tests*100:.1f}%)",
            rf"- **🔴 Baseline Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} > 0.5$):** **`{sub_base_wins}`** ({sub_base_wins/n_tests*100:.1f}%)",
            f"- **⚪ Non-Significant Differences & Exact Ties:** **`{sub_ties}`** ({sub_ties/n_tests*100:.1f}%)",
            "",
            "---",
            "## 🌐 1. Omnibus Kruskal-Wallis H-Test (Group Differences Across Solvers)",
            "",
            "| Problem | Function Class | H-Statistic | p-value | Significant Difference? |",
            "| :--- | :--- | :---: | :---: | :---: |"
        ]
        
        for _, r in df_omnibus_subset.iterrows():
            diff_badge = "🟢 **Yes**" if r['Significant Difference'] == 'Yes' else "⚪ No"
            lines.append(f"| `{r['Problem Name']}` | {r['Function Class']} | {r['H-Statistic']:.3f} | {r['p-value']:.2e} | {diff_badge} |")
            
        lines.extend([
            "",
            "---",
            "## 🔬 2. Pairwise Comparisons (LLaMEA Champion vs. Baselines)",
            "",
            r"*Two-sided Mann-Whitney U tests paired with Vargha-Delaney $\hat{A}_{12}$ effect sizes (where $\hat{A}_{12} < 0.5$ indicates LLaMEA superiority).*",
            "",
            r"| Problem | Function Class | Baseline | LLaMEA Median | Baseline Median | MW p-val | $\hat{A}_{12}$ | Magnitude | Outcome |",
            "| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :--- |"
        ])
        
        prev_prob = None
        for _, r in df_subset.iterrows():
            curr_prob = r['Problem Name']
            if curr_prob != prev_prob:
                prob_str = f"**{curr_prob}**"
                class_str = r['Function Class']
                prev_prob = curr_prob
            else:
                prob_str = ""
                class_str = ""
                
            outcome_str = r['Outcome']
            if 'LLaMEA Wins' in outcome_str:
                badge = f"🟢 **{outcome_str}**"
            elif 'Wins (Sig)' in outcome_str:
                badge = f"🔴 **{outcome_str}**"
            else:
                badge = f"⚪ {outcome_str}"
                
            lines.append(
                f"| {prob_str} | {class_str} | **{r['Baseline']}** | "
                f"{r['LLaMEA Med Res']:.2e} | {r['Baseline Med Res']:.2e} | "
                f"{r['Mann-Whitney p-val']:.2e} | "
                f"{r['A12 Effect Size']:.3f} | {r['Effect Magnitude']} | {badge} |"
            )
            
        sub_md_path.write_text('\n'.join(lines), encoding='utf-8')
        print(f"  📄 Saved per-condition report: {sub_md_path.relative_to(PROJECT_ROOT)}")

    # ── 2. Export Master Consolidated Reports ─────────────────────────────────
    master_csv_path = REPORTS_DIR / 'statistical_summary.csv'
    df_all_pairwise.to_csv(master_csv_path, index=False)
    
    master_omnibus_csv = REPORTS_DIR / 'kruskal_wallis_summary.csv'
    df_all_omnibus.to_csv(master_omnibus_csv, index=False)
    
    master_md_path = REPORTS_DIR / 'statistical_summary.md'
    total_comparisons = len(df_all_pairwise)
    llamea_sig_wins   = len(df_all_pairwise[df_all_pairwise['Outcome'].str.contains('LLaMEA Wins', regex=False)])
    baseline_sig_wins = len(df_all_pairwise[df_all_pairwise['Outcome'].str.contains('Wins (Sig)', regex=False) & ~df_all_pairwise['Outcome'].str.contains('LLaMEA', regex=False)])
    non_sig_ahead     = total_comparisons - llamea_sig_wins - baseline_sig_wins
    
    master_lines = [
        "# 📊 Comprehensive Benchmark & Statistical Analysis Report",
        "",
        "> **Master Roll-Up Report** evaluating **LLaMEA Champion** against classical baseline optimizers (**CMA-ES**, **DE**, **PSO**).",
        "",
        "## 🏆 1. Executive Summary & Win-Loss Metrics",
        "",
        f"- **Total Pairwise Hypothesis Tests ($N$):** `{total_comparisons}`",
        rf"- **🟢 LLaMEA Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} < 0.5$):** **`{llamea_sig_wins}`** ({llamea_sig_wins/total_comparisons*100:.1f}%)",
        rf"- **🔴 Baseline Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} > 0.5$):** **`{baseline_sig_wins}`** ({baseline_sig_wins/total_comparisons*100:.1f}%)",
        f"- **⚪ Non-Significant Differences & Exact Ties:** **`{non_sig_ahead}`** ({non_sig_ahead/total_comparisons*100:.1f}%)",
        "",
        "---",
        "## 📁 2. Direct Links to Per-Condition Statistical Reports",
        "",
        "| Dimension | Noise Level | Condition Label | Dedicated Report Folder |",
        "| :---: | :---: | :--- | :--- |"
    ]
    
    for (dim, noise_std) in sorted(df_all_pairwise.groupby(['Dim', 'Noise Std']).groups.keys()):
        label = "Clean (std = 0.0)" if noise_std == 0.0 else f"Noisy (std = {noise_std})"
        rel_dir = f"{dim}D/std_{noise_std}"
        master_lines.append(f"| **{dim}D** | `{noise_std}` | {label} | [`results/reports/{rel_dir}/statistical_summary.md`]({rel_dir}/statistical_summary.md) |")
        
    master_md_path.write_text('\n'.join(master_lines), encoding='utf-8')
    print(f"\n✨ Master report exported to: {master_md_path.relative_to(PROJECT_ROOT)}")
else:
    print("⚠️ No pairwise benchmark results available to export.")


📁 CSV Report exported to: results/reports/statistical_summary.csv
📁 Omnibus CSV exported to: results/reports/kruskal_wallis_summary.csv
📄 Markdown Report exported to: results/reports/statistical_summary.md


<>:27: SyntaxWarning: invalid escape sequence '\h'
<>:28: SyntaxWarning: invalid escape sequence '\h'
<>:49: SyntaxWarning: invalid escape sequence '\h'
<>:51: SyntaxWarning: invalid escape sequence '\h'
<>:27: SyntaxWarning: invalid escape sequence '\h'
<>:28: SyntaxWarning: invalid escape sequence '\h'
<>:49: SyntaxWarning: invalid escape sequence '\h'
<>:51: SyntaxWarning: invalid escape sequence '\h'
/var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/ipykernel_27662/3757525101.py:27: SyntaxWarning: invalid escape sequence '\h'
  f"- **🟢 LLaMEA Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} < 0.5$):** **`{llamea_sig_wins}`** ({llamea_sig_wins/total_comparisons*100:.1f}%)",
/var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/ipykernel_27662/3757525101.py:28: SyntaxWarning: invalid escape sequence '\h'
  f"- **🔴 Baseline Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} > 0.5$):** **`{baseline_sig_wins}`** ({baseline_sig_wins/total_comparisons*100:.1f}%)",
/var/folde